In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving DataCoSupplyChainDataset.csv to DataCoSupplyChainDataset.csv


In [3]:
df = pd.read_csv(r'/content/DataCoSupplyChainDataset.csv', encoding='latin1')
print(df.shape)

(180519, 53)


In [4]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [6]:
df.drop(columns=['Customer Email', 'Customer Password', 'Latitude', 'Longitude', 'Product Image'], inplace=True)

In [7]:
df.drop(columns=['Order Zipcode', 'Product Description'], inplace=True)

In [8]:
df['Customer name'] = df['Customer Fname'] + ' ' + df['Customer Lname']

In [9]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

In [10]:
df['Order Year'] = df['order date (DateOrders)'].dt.year
df['Order Month'] = df['order date (DateOrders)'].dt.month
df['Order Day'] = df['order date (DateOrders)'].dt.day
df['Order Hour'] = df['order date (DateOrders)'].dt.hour

In [11]:
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

In [12]:
df.drop(columns=['Customer Fname', 'Customer Lname'], inplace=True)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 49 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Type                           180519 non-null  object        
 1   Days for shipping (real)       180519 non-null  int64         
 2   Days for shipment (scheduled)  180519 non-null  int64         
 3   Benefit per order              180519 non-null  float64       
 4   Sales per customer             180519 non-null  float64       
 5   Delivery Status                180519 non-null  object        
 6   Late_delivery_risk             180519 non-null  int64         
 7   Category Id                    180519 non-null  int64         
 8   Category Name                  180519 non-null  object        
 9   Customer City                  180519 non-null  object        
 10  Customer Country               180519 non-null  object        
 11  

In [14]:
df.isnull().sum()

,0
Type,0
Days for shipping (real),0
Days for shipment (scheduled),0
Benefit per order,0
Sales per customer,0
Delivery Status,0
Late_delivery_risk,0
Category Id,0
Category Name,0
Customer City,0


In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
customer_agg = df.groupby('Customer Id').agg(
    Customer_Segment=('Customer Segment', 'first'),
    Frequency=('Order Id', 'nunique'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum')
).reset_index()

In [17]:
segment_summary = customer_agg.groupby('Customer_Segment')[['Total_Sales', 'Total_Profit']].sum()
segment_summary['Margin_pct'] = (segment_summary['Total_Profit'] / segment_summary['Total_Sales'] * 100).round(2)
segment_summary

,Total_Sales,Total_Profit,Margin_pct
Customer_Segment,,,
Consumer,1.909579e+07,2.073488e+06,10.86
Corporate,1.116841e+07,1.202575e+06,10.77
Home Office,6.520538e+06,6.908403e+05,10.59


In [18]:
df.groupby('Customer Segment')['Order Item Quantity'].mean().round(2)

,Order Item Quantity
Customer Segment,
Consumer,2.13
Corporate,2.13
Home Office,2.12


In [19]:
segment_summary = customer_agg.groupby('Customer_Segment').agg(
    Total_Frequency=('Frequency', 'sum'),
    Total_Sales=('Total_Sales', 'sum'),
    Total_Profit=('Total_Profit', 'sum')
)

segment_summary['Margin_pct'] = (segment_summary['Total_Profit'] / segment_summary['Total_Sales'] * 100).round(2)
segment_summary

,Total_Frequency,Total_Sales,Total_Profit,Margin_pct
Customer_Segment,,,,
Consumer,34119,1.909579e+07,2.073488e+06,10.86
Corporate,19856,1.116841e+07,1.202575e+06,10.77
Home Office,11777,6.520538e+06,6.908403e+05,10.59


In [20]:
customer_agg['Frequency'].value_counts().sort_index()

,count
Frequency,
1,8884
2,1363
3,1988
4,2324
5,2172
6,1668
7,1042
8,627
9,325


In [51]:
customer_agg['Final_Segment_v1'] = pd.qcut(
    customer_agg['Total_Score'],
    q=3,
    labels=['At Risk', 'Regular', 'High Value']
)

In [52]:
actual_loss = customer_agg[customer_agg['Total_Profit'] < 0]
caught_correctly = actual_loss[actual_loss['Final_Segment_v1'] == 'At Risk']

catch_rate = len(caught_correctly) / len(actual_loss) * 100
print(catch_rate)

41.58269845170804


In [21]:
freq_dist = customer_agg['Frequency'].value_counts().sort_index()
freq_cumulative_pct = (freq_dist.cumsum() / freq_dist.sum() * 100).round(1)
freq_cumulative_pct

,count
Frequency,
1,43.0
2,49.6
3,59.2
4,70.5
5,81.0
6,89.1
7,94.1
8,97.2
9,98.7


In [22]:
customer_agg['Frequency_Score'] = pd.cut(
    customer_agg['Frequency'],
    bins=[0, 1, 6, 15],
    labels=['Low', 'Medium', 'High']
)

In [23]:
customer_agg['Frequency_Score'].value_counts()

,count
Frequency_Score,
Medium,9515
Low,8884
High,2253


In [24]:
customer_agg['Total_Sales'].describe()

,Total_Sales
count,20652.000000
mean,1781.170589
std,1677.328081
min,11.290000
25%,293.040008
50%,1499.825033
75%,2915.880065
max,10524.170178


In [25]:
customer_agg['Sales_Score'] = pd.qcut(
    customer_agg['Total_Sales'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

In [26]:
customer_agg['Sales_Score'].value_counts()

,count
Sales_Score,
Low,7210
High,6884
Medium,6558


In [27]:
customer_agg['Total_Profit'].describe()

,Total_Profit
count,20652.000000
mean,192.083235
std,389.131248
min,-3868.559982
25%,5.427500
50%,109.830002
75%,397.967494
max,2441.970003


In [28]:
len(customer_agg[customer_agg['Total_Profit'] < 0])

4069

In [29]:
4069 / 20652 * 100

19.70269223319775

In [30]:
loss_customers = customer_agg[customer_agg['Total_Profit'] < 0]
loss_customers['Frequency'].describe()

,Frequency
count,4069.000000
mean,3.149422
std,2.321035
min,1.000000
25%,1.000000
50%,3.000000
75%,5.000000
max,14.000000


In [31]:
profit_customers = customer_agg[customer_agg['Total_Profit'] > 0]
profit_customers['Frequency'].describe()

,Frequency
count,16530.000000
mean,3.199274
std,2.457646
min,1.000000
25%,1.000000
50%,3.000000
75%,5.000000
max,15.000000


In [32]:
loss_customers['Customer_Segment'].value_counts(normalize=True) * 100

,proportion
Customer_Segment,
Consumer,51.634308
Corporate,29.909069
Home Office,18.456623


In [33]:
total_loss = loss_customers['Total_Profit'].sum()
total_profit_all = profit_customers['Total_Profit'].sum()
print(f'Total Loss: {total_loss}')
print(f'Total Profit all: {total_profit_all}')

Total Loss: -1080299.7169197658
Total Profit all: 5047202.690970124


In [34]:
profit_without_losses = total_profit_all - total_loss
print(profit_without_losses)

6127502.40788989


In [35]:
loss_customer_ids = loss_customers['Customer Id']
loss_orders = df[df['Customer Id'].isin(loss_customer_ids)]
loss_orders['Market'].value_counts(normalize=True) * 100

,proportion
Market,
LATAM,28.666629
Europe,28.066011
Pacific Asia,22.809175
USCA,13.951493
Africa,6.506693


In [36]:
customer_agg['Profit_Score'] = pd.qcut(
    customer_agg['Total_Profit'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

In [37]:
customer_agg['Profit_Score'].value_counts()

,count
Profit_Score,
Low,6884
Medium,6884
High,6884


In [38]:
customer_agg[customer_agg['Total_Profit'] < 0]['Profit_Score'].value_counts()

,count
Profit_Score,
Low,4069
Medium,0
High,0


In [44]:
score_map = {'Low': 1, 'Medium': 2, 'High': 3}

customer_agg['Freq_Num'] = customer_agg['Frequency_Score'].map(score_map).astype(int)
customer_agg['Sales_Num'] = customer_agg['Sales_Score'].map(score_map).astype(int)
customer_agg['Profit_Num'] = customer_agg['Profit_Score'].map(score_map).astype(int)

customer_agg['Total_Score'] = (
    customer_agg['Freq_Num'] * 20 +
    customer_agg['Sales_Num'] * 35 +
    customer_agg['Profit_Num'] * 45
)

In [45]:
customer_agg['Total_Score'].value_counts().sort_index()

,count
Total_Score,
100,3942
120,20
135,383
145,3194
155,1132
165,54
175,2
180,1051
190,1026


In [46]:
customer_agg['Final_Segment'] = 'At Risk'
customer_agg.loc[customer_agg['Total_Profit'] >= 0, 'Final_Segment'] = None
profitable = customer_agg[customer_agg['Total_Profit'] >= 0].copy()
profitable['Final_Segment'] = pd.qcut(
    profitable['Total_Score'],
    q=2,
    labels=['Regular', 'High Value']
)

In [47]:
customer_agg.loc[profitable.index, 'Final_Segment'] = profitable['Final_Segment']

In [48]:
customer_agg['Final_Segment'].value_counts()

,count
Final_Segment,
Regular,8742
High Value,7841
At Risk,4069


In [50]:
df_final = df.merge(
    customer_agg[['Customer Id', 'Final_Segment']],
    on='Customer Id',
    how='left'
)

df_final.to_csv('dataco_order_level_with_segment.csv', index=False)

KeyboardInterrupt: 

In [ ]:
customer_agg.to_csv('customer_segmentation_summary.csv', index=False)

In [ ]:
from google.colab import files

files.download('dataco_order_level_with_segment.csv')
files.download('customer_segmentation_summary.csv')